# Orchestrating Multiple Agents

So far each notebook has used **one** agent. Real agentic systems get their power from **teams** — several specialist agents coordinated by an *orchestrator* that decides who speaks next and when the conversation is done.

## What is orchestration?
An **orchestrator** is the policy that answers two questions on every turn:
1. *Who talks next?*
2. *Are we finished?*

AutoGen v0.4 ships several orchestrators in `autogen_agentchat.teams`. We'll start with the simplest:

| Team | Who picks the next speaker | When to use |
|---|---|---|
| `RoundRobinGroupChat` | Fixed turn order | Pipelines with a known sequence (research → write → critique) |

## What we'll cover
1. Build three specialist agents (researcher, writer, critic)
2. Orchestrate them with `RoundRobinGroupChat`
3. Stop the team with **termination conditions**
4. Reset team state between runs

> **Mental model:** an agent is a worker. A team is a worker + a *policy* for who works next. Orchestration is just that policy.

## 1. Setup
Same setup as the earlier notebooks — load the API key and create one shared model client.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found. Add it to .env"

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

## 2. Build three specialist agents

Each agent gets a focused **`system_message`** (its inner monologue) and a clear **`description`** (its résumé — visible to other agents and humans coordinating the team).

We'll build a tiny content pipeline:
- **researcher** — pulls out the key facts to cover
- **writer** — turns those facts into a short draft
- **critic** — reviews the draft and either approves it or asks for a revision

In [ ]:
researcher = AssistantAgent(
    name="researcher",
    model_client=model_client,
    description="Gathers key facts and bullet points about a topic before writing begins.",
    system_message=(
        "You are a researcher. Given a topic, list 3-5 short, factual bullet points "
        "that a writer can use. No prose, just bullets."
    ),
)

writer = AssistantAgent(
    name="writer",
    model_client=model_client,
    description="Turns research bullet points into a short, readable paragraph for a general audience.",
    system_message=(
        "You are a writer. Using the researcher's bullets, write ONE clear paragraph "
        "(4-6 sentences) for a general audience. Do not add new facts."
    ),
)

critic = AssistantAgent(
    name="critic",
    model_client=model_client,
    description="Reviews drafts for clarity and accuracy. Approves with the word APPROVE when satisfied.",
    system_message=(
        "You are an editor. Review the writer's paragraph. "
        "If it is clear, accurate, and 4-6 sentences, reply with the single word: APPROVE. "
        "Otherwise, give ONE specific suggestion in one sentence."
    ),
)

for a in (researcher, writer, critic):
    print(f"{a.name:10s} -> {a.description}")

## 3. Orchestrate with `RoundRobinGroupChat`

The simplest orchestrator: agents speak in the order you list them, looping forever — until a **termination condition** says stop.

Two termination conditions are useful right away:
- **`TextMentionTermination("APPROVE")`** — stop when any agent says the magic word.
- **`MaxMessageTermination(n)`** — hard cap, prevents runaway loops.

Combining them with `|` (OR) means *stop on whichever fires first* — a safety net plus a happy-path exit.

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

team = RoundRobinGroupChat(
    participants=[researcher, writer, critic],
    termination_condition=termination,
)

await Console(team.run_stream(task="Write a short explainer on what an AI agent is."))

Watch the trace: `researcher` → `writer` → `critic`, in order. The team stops as soon as the critic says `APPROVE` — or after 8 messages, whichever comes first.

## 4. Reset before reusing

Teams keep the full conversation history across runs, just like single agents. Before kicking off an unrelated task, reset:

In [ ]:
await team.reset()
print("Team state cleared.")

## 5. Clean up
Always close the model client when done.

In [ ]:
await model_client.close()
print("Done.")

## Recap

- A **team** is agents + an **orchestrator** that decides who speaks next and when to stop.
- **`RoundRobinGroupChat`** — fixed order. Use for known pipelines.
- **Termination conditions** (`TextMentionTermination`, `MaxMessageTermination`, combined with `|`) are how teams know to stop.
- Call `await team.reset()` between unrelated runs.

### Try it yourself
1. Add a fourth agent — say, a `fact_checker` — to the participants list and watch where it lands in the rotation.
2. Replace `MaxMessageTermination` with `TokenUsageTermination` to cap cost instead of message count.
3. Tighten the critic's `system_message` so it's harder to please, and watch the loop run longer.